# Prompt mu, fine-tune mu — aynı taban, tek değişken

Bu notebook **eğitmiyor**. Bir soruyu cevaplıyor: `grounded_format`'ı bir
fine-tune olmadan, sadece talimatı düzelterek yükseltebilir miyiz?

Soru ucuz değil, gerekli. `persona-measure` şunu ölçtü — `Qwen3-4B-Instruct-2507`
tabanı, 100 satırlık validation seti:

| metrik | taban | ne diyor |
|---|---:|---|
| `citation_valid` | 1.00 | öğretilecek bir şey yok |
| `asked_when_thin` | 19/28 | öğretilecek bir şey yok — **korunacak taban** |
| `grounded_format` | **0.64** | tek gerçek açık, ve bir **biçim** sorunu |
| `decision_match` | 15/72 | düşük, ama bankadan ayrılmadan hedef sayılmaz |

Ve bu hattın ilk adapter'ı `grounded_format`'ı 1.00'a çıkarırken
`asked_when_thin`'i **19/28 → 0/28** yaptı. Üç saatlik bir koşu, ürünün var olma
sebebi olan davranışı yok etti.

Prompt'un bedeli sıfır. Kazanıyorsa o adapter'a hiç gerek yok.

## Deney

İki taraf, **aynı model**, **aynı kanıt**, **aynı yer gerçeği**. Değişen tek şey
system turn'ü:

| taraf | system prompt |
|---|---|
| `v1` | setin içinde geldiği hâli — backend'in bugün gönderdiği |
| `v2` | `prompts/persona_v2.txt` |

`--system-prompt-file` yalnız system turn'ünü değiştiriyor ve satır başına tam
bir tane bulamazsa çıkıyor; bulamadığı hâlde karşılaştırma tek değişkenli
olmaktan çıkardı.

**v2'nin üç değişikliği**, hepsi `v1`'de ölçülen kusura karşılık geliyor:

1. **Biçim şartı artık son söz.** `v1` formatı verdikten sonra bir üslup
   paragrafı daha ekliyor, yani modelin okuduğu son talimat çıktı sözleşmesi
   değil. `v2`'de son satır "Bu üç satır cevabının son satırları olmalı."
2. **İki mod isimlendirildi ve ayrıldı** — `BİÇİM 1 (SORU)` / `BİÇİM 2 (KARAR)`,
   aralarında "üçüncü bir biçim yok, ikisini karıştırma". Adapter'ın kaybettiği
   ayrım tam buydu ve `v1`'de hiç açıkça yazmıyor.
3. **Soru modunda ne yazılmayacağı söylendi.** `v1` "sor ve dur" diyor ama KARAR
   satırını yasaklamıyor; `asked_when_thin` tam olarak onu cezalandırıyor.

## Karar kuralı

`grounded_format` yükselmeli **ve** `asked_when_thin` düşmemeli. İkincisi
düşerse `v2` reddedilir: `v1`'in 0.64'ü, soru sormayı kaybetmiş bir 0.95'ten
iyidir. Bu, adapter'ı yayına almayan kuralın aynısı.

## Bütçe

`persona-measure`'ın ölçülen hızından: 100 satır + model yüklemesi ≈ 20 dk.

| aşama | tahmin |
|---|---|
| kurulum + model indirme | ~11 dk |
| v1, 100 satır | ~20 dk |
| v2, 100 satır | ~20 dk |
| **toplam** | **~51 dk** |

Eğitim yok, deadline yok.

In [ ]:
import glob, json, os, shutil, subprocess, sys
import torch

assert torch.cuda.is_available(), "GPU acik degil - Settings > Accelerator > GPU T4"
cap = torch.cuda.get_device_capability(0)
print("GPU:", torch.cuda.get_device_name(0), "sm_%d%d" % cap)
assert cap >= (7, 5), f"sm_{cap[0]}{cap[1]} yetersiz - T4 (sm_75) gerekiyor"

In [ ]:
!pip -q install -U "transformers>=4.51" "peft>=0.11" "bitsandbytes>=0.43" "accelerate>=0.30" datasets 2>&1 | tail -2
import transformers, peft
print("transformers", transformers.__version__, "| peft", peft.__version__)
# torchao kaldiriliyor, yukseltilmiyor: peft'in LoRA dispatcher'i kuantize
# OLMAYAN her Linear icin is_torchao_available() soruyor ve uyumsuz surumde
# ImportError firlatiyor. Bu notebook fp16 yukluyor, yani o kola giriliyor.
!pip -q uninstall -y torchao 2>&1 | tail -1

In [ ]:
def find_mount(slug, marker):
    hits = [p for p in glob.glob(f"/kaggle/input/**/{marker}", recursive=True)
            if slug.split("/")[-1] in p]
    assert hits, f"'{slug}' bagli degil (aranan: {marker})"
    return os.path.dirname(sorted(hits, key=len)[0])


WORK = "/kaggle/working"
DATA = find_mount("emrahik/persona-dataset", "persona_eval.jsonl")
print("veri seti:", DATA)

os.makedirs(f"{WORK}/data", exist_ok=True)
os.makedirs(f"{WORK}/prompts", exist_ok=True)
for f in os.listdir(DATA):
    if f.endswith(".jsonl"):
        shutil.copy(f"{DATA}/{f}", f"{WORK}/data/{f}")
    elif f.endswith(".txt"):
        shutil.copy(f"{DATA}/{f}", f"{WORK}/prompts/{f}")
    else:
        shutil.copy(f"{DATA}/{f}", f"{WORK}/{f}")
os.chdir(WORK)
print(sorted(os.listdir(WORK)), sorted(os.listdir("prompts")))

h = subprocess.run([sys.executable, "persona_eval.py", "--help"],
                   capture_output=True, text=True).stdout
for flag in ("--system-prompt-file", "--base-only", "--local"):
    assert flag in h, (f"persona_eval.py '{flag}' bilmiyor — Kaggle dataset'i "
                       f"eski. push_persona.sh calistir, sonra yeniden push et.")
assert os.path.exists("prompts/persona_v2.txt"), \
    "prompts/persona_v2.txt dataset'te yok — push_persona.sh onu tasiyor mu?"
print("script ve prompt guncel")

## v1 — setin içinde geldiği prompt

Bu taraf `persona-measure`'ın taban ölçümünün tekrarı ve **kasten öyle**: aynı
oturumda, aynı kütüphane sürümleriyle ölçülmüş bir referans olmadan v2'nin
farkının ne kadarının prompt olduğu söylenemez. Sayı `persona-measure`'ınkine
yakın çıkmazsa, ölçümde prompt dışında değişen bir şey var demektir ve
karşılaştırma o hâlde okunmaz.

In [ ]:
LIMIT = 100
BASE = "Qwen/Qwen3-4B-Instruct-2507"

r = subprocess.run([sys.executable, "persona_eval.py",
                    "--local", "--base-only",
                    "--local-base-model", BASE,
                    "--eval", "data/persona_eval.jsonl",
                    "--meta", "data/persona_eval_meta.jsonl",
                    "--limit", str(LIMIT),
                    "--out", "out/prompt_v1.json"])
assert r.returncode == 0, f"v1 olcumu coktu (exit {r.returncode})"
v1 = json.load(open("out/prompt_v1.json"))["before"]
print(json.dumps({k: v for k, v in v1.items() if k != "samples"}, indent=2))

## v2 — düzeltilmiş prompt

In [ ]:
r = subprocess.run([sys.executable, "persona_eval.py",
                    "--local", "--base-only",
                    "--local-base-model", BASE,
                    "--system-prompt-file", "prompts/persona_v2.txt",
                    "--eval", "data/persona_eval.jsonl",
                    "--meta", "data/persona_eval_meta.jsonl",
                    "--limit", str(LIMIT),
                    "--out", "out/prompt_v2.json"])
assert r.returncode == 0, f"v2 olcumu coktu (exit {r.returncode})"
v2 = json.load(open("out/prompt_v2.json"))["before"]
print(json.dumps({k: v for k, v in v2.items() if k != "samples"}, indent=2))

In [ ]:
n_clarify = sum(1 for l in open("data/persona_eval_meta.jsonl")
                if json.loads(l)["mode"] == "clarify")
n_decide = LIMIT - n_clarify
keys = ("citation_valid", "grounded_format", "asked_when_thin", "decision_match")
den = {"citation_valid": LIMIT, "grounded_format": n_decide,
       "asked_when_thin": n_clarify, "decision_match": n_decide}

print(f"\n{LIMIT} satir  ({n_decide} decide / {n_clarify} clarify)\n")
print(f"{'metrik':<20}{'v1':>8}{'v2':>8}{'delta':>9}   payda")
print("-" * 56)
for k in keys:
    a, b = v1.get(k), v2.get(k)
    f = lambda v: "n/a" if v is None else f"{v:.2f}"
    d = "-" if (a is None or b is None) else f"{b - a:+.2f}"
    print(f"{k:<20}{f(a):>8}{f(b):>8}{d:>9}   {den[k]} satir")

# Ayni karar kurali adapter'i yayina almayan kural. Bir biciM kazanci, soru
# sorma davranisinin yerine gecmez.
gf_up = (v1["grounded_format"] is not None and v2["grounded_format"] is not None
         and v2["grounded_format"] > v1["grounded_format"])
ask_held = (v1["asked_when_thin"] is not None and v2["asked_when_thin"] is not None
            and v2["asked_when_thin"] >= v1["asked_when_thin"])

print("\n" + "=" * 56)
if gf_up and ask_held:
    print("v2 KAZANDI: grounded_format yukseldi, asked_when_thin korundu.")
    print("Sonraki adim bir egitim kosusu DEGIL — v2'yi backend'in")
    print("internal/decision prompt'una tasi. Cikarim onu oradan okuyor;")
    print("burada kazanan bir dosya, gonderilmedigi surece kazanmis degil.")
elif ask_held:
    print("v2 grounded_format'i yukseltmedi ama tabani da bozmadi.")
    print("Prompt yolu bu haliyle yetmiyor; ya baska bir varyant, ya egitim —")
    print("ve egitim ise clarify satirlari agirliklandirilarak.")
else:
    print("v2 REDDEDILDI: asked_when_thin dustu.")
    print("v1'in 0.64'u, soru sormayi kaybetmis bir 0.95'ten iyidir.")
    print("Adapter'i yayina almayan kuralin aynisi.")
print("=" * 56)

# Ciktilar da bakilabilsin: rate neyin oldugunu soyler, ornek nedenini.
for name, side in (("v1", v1), ("v2", v2)):
    bad = [s for s in side.get("samples", []) if any(v is False for v in s["scores"].values())]
    if bad:
        s = bad[0]
        print(f"\n--- {name}, basarisiz ornek (satir {s['row']}, mode={s['mode']}) ---")
        print(json.dumps(s["scores"]))
        print(s["answer"][:400])